In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
import uuid
from sentence_transformers import SentenceTransformer
from qdrant_client.models import Filter, FieldCondition, MatchValue, Vector

In [3]:
# Load a simple transformer model that converts text → vector
model = SentenceTransformer("all-MiniLM-L6-v2")

# Your mini database of facts
texts = [
    "Qdrant is a vector database.",
    "LangChain is used to build LLM apps.",
    "SentenceTransformers convert text into embeddings.",
]

# Turn each sentence into a vector
vectors = model.encode(texts).tolist()

In [ ]:
# Start Qdrant client in memory (no Docker needed for now)
# Can be run in Memory (ends when script ends)
# Can be run as Server Locally via Docker (survives restarts)
# Can be run on Qdrant Cloud 
client = QdrantClient(":memory:") 

# Create a collection (like a table for vectors)
client.recreate_collection(
    collection_name="my_test_collection",
    vectors_config=VectorParams(size=len(vectors[0]), distance=Distance.COSINE)

)

# Upload your vectors to Qdrant
# PointStruct ---> defines ONE point (Vector + Metadata)
points = [
    PointStruct(
        id=str(uuid.uuid4()), # unique ID
        vector=vec, # list of fixed-size floats
        payload={"text": text} # dictionary of metadata for filtering
    )
    for vec, text in zip(vectors, texts)
]

#  upsert = update OR insert
# if ID exists, update point
# if ID new, insert
client.upsert(collection_name="my_test_collection", points=points) 

/var/folders/y5/gm5w6cr53lq9bx_hyjyfb7nr0000gn/T/ipykernel_1418/2819266760.py:5: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


UpdateResult(operation_id=0, status=<UpdateStatus.COMPLETED: 'completed'>)

In [8]:
# Create a query
query = "What is Qdrant?"

# Embed the query
query_vector = model.encode(query).tolist()

# Search Qdrant for similar vectors
results = client.search(
    collection_name="my_test_collection", # name of collection
    query_vector=query_vector, # embedded query
    limit=2  # top 2 most similar
)

# Show results
for res in results:
    print(f"Score: {res.score:.4f}")
    print("Matched Text:", res.payload["text"])
    print("-----")


Score: 0.6877
Matched Text: Qdrant is a vector database.
-----
Score: 0.2242
Matched Text: LangChain is used to build LLM apps.
-----


/var/folders/y5/gm5w6cr53lq9bx_hyjyfb7nr0000gn/T/ipykernel_1418/620320426.py:8: DeprecationWarning: `search` method is deprecated and will be removed in the future. Use `query_points` instead.
  results = client.search(


In [ ]:
filter_example = Filter(
    must=[
        FieldCondition(key="text", # text metadata must match 'match' sentence
                       match=MatchValue(value="Qdrant is a vector database."))
    ]
)

results_filtered = client.query_points(
    collection_name="my_test_collection", # name of collection
    query=query_vector, # vectorized query
    limit=3, #top 3 results
    query_filter=filter_example, #what is the filter to be used
)

# query response object returned by Qdrant
# .points is list of matched results/ScoredPoint objects from query
# Each point has vectorID, sim score, metadata,
points_list = results_filtered.points


print("\nResults with filter:")
for res in points_list:
    print(f"Score: {res.score:.4f}") # sim score
    print("Matched Text:", res.payload.get("text", "No text")) # metadata dict
    print("---") # separates score visually in output



Results with filter:
Score: 0.6877
Matched Text: Qdrant is a vector database.
---
